## rag(retrival augmented generation) from scratch

### ingestion pipeline

In [17]:
# loading text/data from a file(pdf)
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

dir_loader = DirectoryLoader("../data/", glob="**/*.pdf", loader_cls=PyPDFLoader)
documents = dir_loader.load()
print(documents)

[Document(metadata={'producer': 'LibreOffice 24.2.7.2', 'creator': 'LibreOffice 24.2.7.2', 'creationdate': '2026-10-09T20:41:34+05:30', 'title': 'Three_Body_Problem_Netflix_Season_1_Comprehensive_Guide.docx', 'source': '../data/document_3dp.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1'}, page_content='The story at a glance\nThree Body Problem begins with a crisis in modern physics: scientists are dying, experiments are \nbehaving impossibly, and a mysterious countdown appears to selected researchers. The mystery \neventually leads back to Ye Wenjie, an astrophysicist whose faith in humanity was shattered during\nChina’s Cultural Revolution.\nDecades earlier, Wenjie answered a signal from an extraterrestrial civilization. She had received a \nwarning not to reply, because doing so would reveal Earth’s location to a civilization seeking a \nhabitable world. She answered anyway, believing an outside power might force humanity to \nchange.\nThe aliens—called the San-Ti in the Netfl

In [18]:
# creating chunks from text loaded

from langchain_text_splitters import RecursiveCharacterTextSplitter


def process_and_chunk(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks.")
    return split_docs


chunks = process_and_chunk(documents)
# chunks

split 17 documents into 53 chunks.


In [19]:
# embedding
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, PreTrainedModel
import numpy as np
from typing import List, Any, Dict


class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"loading embedding manager: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"model loaded with embedding dimentions: {self.model.get_embedding_dimension()}"
            )
        except Exception as e:
            print(f"error loading model {self.model_name}: {e}")
            raise

    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("model not loaded!")
        print(f"generating embedding for {len(texts)} texts..")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"generated embeddings with shape: {embeddings.shape}")
        return embeddings


embedding_manager = EmbeddingManager()
embedding_manager

loading embedding manager: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1620.41it/s]


model loaded with embedding dimentions: 384


In [20]:
# vector store
import os
import chromadb
import hashlib
import numpy as np

class VectorManager:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._load_store()

    def _load_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                   "description": "pdf documents for rag",
                    "hnsw:space": "cosine",
                    },)
            print(f"vector store initialized, collection: {self.collection_name}")
            print(f"existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"error loading vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("no of documents does not match no of embedding")

        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            # Deterministic ID: hash of source + page + chunk index
            source = doc.metadata.get('source', 'unknown')
            page = doc.metadata.get('page', 0)
            hash_input = f"{source}_{page}_{i}".encode()
            doc_id = hashlib.md5(hash_input).hexdigest()[:12]
            ids.append(doc_id
            )

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        try:
            self.collection.upsert(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"added {len(documents)} documents into vector store")
            print(f"total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"error adding documents to vector store: {e}")
            raise

vector_store=VectorManager()
vector_store

vector store initialized, collection: pdf_documents
existing documents in collection: 203


In [21]:
# convert text into embeddings
texts = [doc.page_content for doc in chunks]

# generate embedding
embeddings = embedding_manager.generate_embedding(texts)

# store in vector db
vector_store.add_documents(chunks, embeddings)

generating embedding for 53 texts..


Batches: 100%|██████████| 2/2 [00:07<00:00,  3.80s/it]


generated embeddings with shape: (53, 384)
added 53 documents into vector store
total documents in collection: 252


### retrieval pipeline

In [22]:
class RagRetriever:
    def __init__(self, vector_store: VectorManager, embedding_manager=EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self, query: str, top_k: int = 5, score_threshold: float = 0.0
    ) -> List[Dict[str, any]]:
        print(f"retrieving documents for query: {query}")
        print(f"top k: {top_k}, score threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()], n_results=top_k
            )

            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append(
                            {
                                "id": doc_id,
                                "content": document,
                                "metadata": metadata,
                                "similarity_score": similarity_score,
                                "distance": distance,
                                "rank": i + 1,
                            }
                        )
                print(f"retireved {len(retrieved_docs)} documents after search")
            else:
                print("no documents found")
            return retrieved_docs

        except Exception as e:
            print(f"error during retrieval: {e}")


rag_retriever = RagRetriever(vector_store, embedding_manager)
rag_retriever

In [27]:
def rag_simple(query, retriever, top_k=5):
    results = retriever.retrieve(query, top_k=top_k)
    if not results:
        return "no relevant context found"

    context = "\n\n".join(d["content"] for d in results)
    prompt = f"""Answer the question using only the context below.
Give a complete, detailed answer. If the context doesn't contain the answer, say so.

Context:
{context}

Question: {query}
Answer:"""

    response = chat(
        model="qwen3:0.6b",  # or llama3.2:3b
        messages=[{"role": "user", "content": prompt}],
        options={"num_ctx": 8192, "temperature": 0.2},
    )
    return response.message.content